In [1]:
import pandas as pd
import os
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import PassiveAggressiveClassifier
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchtext.vocab as vocab
from torch.autograd import Variable


In [2]:
!conda install -y gdown

!gdown --folder 15Wn46r7gidaiZbx2ArFYsd7rjYH4y7JM

/bin/bash: line 1: conda: command not found
Retrieving folder list
Processing file 1YvkO_dP-om5Yh-EJuP2-fljnIEfP-42C README
Processing file 1WJyzjqaEHUBLzijbjyjpzurPuLaZOSyG test.tsv
Processing file 1RvXY274ln1cKZR6LaUacyytgrrxNqH3K train.tsv
Processing file 1n3TWKuZx4ot2zsFnjTEZTnrrtcsZHCKi valid.tsv
Retrieving folder list completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=1YvkO_dP-om5Yh-EJuP2-fljnIEfP-42C
To: /content/liar_dataset/README
100% 1.67k/1.67k [00:00<00:00, 6.30MB/s]
Downloading...
From: https://drive.google.com/uc?id=1WJyzjqaEHUBLzijbjyjpzurPuLaZOSyG
To: /content/liar_dataset/test.tsv
100% 301k/301k [00:00<00:00, 95.5MB/s]
Downloading...
From: https://drive.google.com/uc?id=1RvXY274ln1cKZR6LaUacyytgrrxNqH3K
To: /content/liar_dataset/train.tsv
100% 2.41M/2.41M [00:00<00:00, 273MB/s]
Downloading...
From: https://drive.google.com/uc?id=1n3TWKuZx4ot2zsFnjTEZTnrrtcsZHCKi
To: /content/liar_dataset/va

In [ ]:

#open the csv files
csv_path_fake = os.path.join( './', 'News_dataset', 'Fake.csv')
csv_path_true = os.path.join( './', 'News_dataset', 'True.csv')

df_fake = pd.read_csv(csv_path_fake)
df_true = pd.read_csv(csv_path_true)


In [ ]:
#add a label column to the dataframes
df_fake['label'] = 'fake'
df_true['label'] = 'true'

df = pd.concat([df_fake, df_true], ignore_index=True)

#save the dataframe to a csv file
df.to_csv('news.csv', index=False)

In [ ]:
# open the csv file with pandas

df = pd.read_csv('news.csv')

#split the data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(df['text'], df['label'], test_size=0.1, random_state=7)

#initialize a TfidfVectorizer
tfidf_vectorizer = TfidfVectorizer(stop_words='english', max_df=0.9)

#fit and transform train set, transform test set
tfidf_train = tfidf_vectorizer.fit_transform(X_train)
tfidf_test = tfidf_vectorizer.transform(X_test)

In [ ]:
#initialize a PassiveAggressiveClassifier
pac = PassiveAggressiveClassifier(max_iter=50)
pac.fit(tfidf_train, y_train)

#predict on the test set and calculate accuracy
y_pred = pac.predict(tfidf_test)
score = accuracy_score(y_test, y_pred)
print(f'Accuracy: {round(score*100,2)}%')

#build confusion matrix
confusion_matrix(y_test, y_pred, labels=['fake', 'true'])


Accuracy: 99.38%


array([[2335,   11],
       [  17, 2127]])

In [ ]:
#with logistic regression
logreg = LogisticRegression()
logreg.fit(tfidf_train, y_train)
y_pred = logreg.predict(tfidf_test)
score = accuracy_score(y_test, y_pred)
print(f'Accuracy: {round(score*100,2)}%')

#build confusion matrix
confusion_matrix(y_test, y_pred, labels=['fake', 'true'])

Accuracy: 98.66%


array([[2317,   29],
       [  31, 2113]])

In [ ]:
#with multinomial naive bayes

nb = MultinomialNB()
nb.fit(tfidf_train, y_train)
y_pred = nb.predict(tfidf_test)
score = accuracy_score(y_test, y_pred)
print(f'Accuracy: {round(score*100,2)}%')

#build confusion matrix
confusion_matrix(y_test, y_pred, labels=['fake', 'true'])

Accuracy: 93.96%


array([[2221,  125],
       [ 146, 1998]])

In [ ]:
# with a random forest classifier

rf = RandomForestClassifier()
rf.fit(tfidf_train, y_train)
y_pred = rf.predict(tfidf_test)
score = accuracy_score(y_test, y_pred)
print(f'Accuracy: {round(score*100,2)}%')

#build confusion matrix
confusion_matrix(y_test, y_pred, labels=['fake', 'true'])

Accuracy: 98.89%


array([[2324,   22],
       [  28, 2116]])

In [ ]:
# with a support vector machine

svc = LinearSVC(random_state=0, dual='auto')
svc.fit(tfidf_train, y_train)
y_pred = svc.predict(tfidf_test)
score = accuracy_score(y_test, y_pred)
print(f'Accuracy: {round(score*100,2)}%')

#build confusion matrix
confusion_matrix(y_test, y_pred, labels=['fake', 'true'])

Accuracy: 99.47%


array([[2337,    9],
       [  15, 2129]])

In [3]:
# using Liar dataset

csv_path_liar_train = os.path.join( '/content', 'liar_dataset', 'train.tsv')
csv_path_liar_test = os.path.join( '/content', 'liar_dataset', 'test.tsv')

df_liar_train = pd.read_csv(csv_path_liar_train, sep='\t', header=None)
df_liar_test = pd.read_csv(csv_path_liar_test, sep='\t', header=None)

df_liar_train.columns = ['id', 'label', 'statement', 'subject', 'speaker', 'job', 'state', 'party', 'barely_true', 'false', 'half_true', 'mostly_true', 'pants_on_fire', 'context']
df_liar_test.columns = ['id', 'label', 'statement', 'subject', 'speaker', 'job', 'state', 'party', 'barely_true', 'false', 'half_true', 'mostly_true', 'pants_on_fire', 'context']
tfidf_vectorizer = TfidfVectorizer(stop_words='english', max_df=0.9)

#fit and transform train set, transform test set
tfidf_train = tfidf_vectorizer.fit_transform(df_liar_train['statement'])
tfidf_test = tfidf_vectorizer.transform(df_liar_test['statement'])

In [ ]:


#initialize a PassiveAggressiveClassifier
pac = PassiveAggressiveClassifier(max_iter=1000)
pac.fit(tfidf_train, df_liar_train['label'])

#predict on the test set and calculate accuracy
y_pred = pac.predict(tfidf_test)
score = accuracy_score(df_liar_test['label'], y_pred)
print(f'Accuracy: {round(score*100,2)}%')

#build confusion matrix
confusion_matrix(df_liar_test['label'], y_pred, labels=['true', 'mostly-true', 'half-true', 'barely-true', 'false', 'pants-fire'])


Accuracy: 22.81%


array([[43, 38, 49, 22, 47,  9],
       [49, 51, 52, 36, 38, 15],
       [50, 58, 59, 39, 37, 22],
       [36, 43, 31, 40, 41, 21],
       [38, 49, 38, 36, 74, 14],
       [14, 11, 13, 15, 17, 22]])

In [ ]:
# with logistic regression
logreg = LogisticRegression(max_iter=1000)
logreg.fit(tfidf_train, df_liar_train['label'])
y_pred = logreg.predict(tfidf_test)
score = accuracy_score(df_liar_test['label'], y_pred)
print(f'Accuracy: {round(score*100,2)}%')

#build confusion matrix
confusion_matrix(df_liar_test['label'], y_pred, labels=['true', 'mostly-true', 'half-true', 'barely-true', 'false', 'pants-fire'])


Accuracy: 25.02%


array([[47, 50, 46, 13, 50,  2],
       [50, 56, 65, 25, 44,  1],
       [28, 65, 84, 37, 47,  4],
       [22, 37, 53, 44, 55,  1],
       [29, 50, 57, 26, 84,  3],
       [ 7, 14, 25, 13, 31,  2]])

In [ ]:
# with multinomial naive bayes

nb = MultinomialNB()
nb.fit(tfidf_train, df_liar_train['label'])
y_pred = nb.predict(tfidf_test)
score = accuracy_score(df_liar_test['label'], y_pred)
print(f'Accuracy: {round(score*100,2)}%')

#build confusion matrix
confusion_matrix(df_liar_test['label'], y_pred, labels=['true', 'mostly-true', 'half-true', 'barely-true', 'false', 'pants-fire'])

Accuracy: 23.76%


array([[ 15,  59,  79,   3,  52,   0],
       [ 14,  65, 115,   7,  40,   0],
       [  7,  68, 126,  17,  47,   0],
       [  8,  37,  93,  18,  56,   0],
       [ 12,  51,  99,  10,  77,   0],
       [  2,  18,  36,   7,  29,   0]])

In [ ]:
# with a random forest classifier

rf = RandomForestClassifier()
rf.fit(tfidf_train, df_liar_train['label'])
y_pred = rf.predict(tfidf_test)
score = accuracy_score(df_liar_test['label'], y_pred)
print(f'Accuracy: {round(score*100,2)}%')

#build confusion matrix
confusion_matrix(df_liar_test['label'], y_pred, labels=['true', 'mostly-true', 'half-true', 'barely-true', 'false', 'pants-fire'])

Accuracy: 24.63%


array([[ 23,  61,  40,  15,  67,   2],
       [ 35,  76,  48,  26,  55,   1],
       [ 18,  76,  70,  21,  77,   3],
       [ 15,  46,  42,  30,  76,   3],
       [ 20,  43,  48,  26, 109,   3],
       [ 10,  10,  14,  13,  41,   4]])

In [ ]:
# with a support vector machine

svc = LinearSVC(random_state=0, dual='auto')
svc.fit(tfidf_train, df_liar_train['label'])
y_pred = svc.predict(tfidf_test)
score = accuracy_score(df_liar_test['label'], y_pred)
print(f'Accuracy: {round(score*100,2)}%')

#build confusion matrix
confusion_matrix(df_liar_test['label'], y_pred, labels=['true', 'mostly-true', 'half-true', 'barely-true', 'false', 'pants-fire'])

Accuracy: 24.55%


array([[53, 44, 51, 15, 38,  7],
       [52, 49, 53, 36, 42,  9],
       [31, 68, 67, 51, 38, 10],
       [24, 31, 47, 56, 44, 10],
       [35, 47, 46, 30, 76, 15],
       [10, 12, 21, 17, 22, 10]])

In [4]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

cuda


In [ ]:
class Net(nn.Module):

    def __init__(self,

                 statement_vocab_dim,
                 subject_vocab_dim,
                 speaker_vocab_dim,
                 speaker_pos_vocab_dim,
                 state_vocab_dim,
                 party_vocab_dim,
                 context_vocab_dim,

                 statement_embed_dim = 100,
                 statement_kernel_num = 14,
                 statement_kernel_size = [3, 4, 5],

                 subject_embed_dim = 5,
                 subject_lstm_nlayers = 2,
                 subject_lstm_bidirectional = True,
                 subject_hidden_dim = 5,

                 speaker_embed_dim = 5,

                 speaker_pos_embed_dim = 10,
                 speaker_pos_lstm_nlayers = 2,
                 speaker_pos_lstm_bidirectional = True,
                 speaker_pos_hidden_dim = 5,

                 state_embed_dim = 5,

                 party_embed_dim = 5,

                 context_embed_dim = 20,
                 context_lstm_nlayers = 2,
                 context_lstm_bidirectional = True,
                 context_hidden_dim = 6,
                 dropout = 0.5):

        # Statement CNN
        super(Net, self).__init__()

        self.statement_vocab_dim = statement_vocab_dim
        self.statement_embed_dim = statement_embed_dim
        self.statement_kernel_num = statement_kernel_num
        self.statement_kernel_size = statement_kernel_size

        self.statement_embedding = nn.Embedding(self.statement_vocab_dim, self.statement_embed_dim)
        self.statement_convs = [nn.Conv2d(1, self.statement_kernel_num, (kernel_, self.statement_embed_dim)) for kernel_ in self.statement_kernel_size]

        # Subject
        self.subject_vocab_dim = subject_vocab_dim
        self.subject_embed_dim = subject_embed_dim
        self.subject_lstm_nlayers = subject_lstm_nlayers
        self.subject_lstm_num_direction = 2 if subject_lstm_bidirectional else 1
        self.subject_hidden_dim = subject_hidden_dim

        self.subject_embedding = nn.Embedding(self.subject_vocab_dim, self.subject_embed_dim)
        self.subject_lstm = nn.LSTM(
            input_size = self.subject_embed_dim,
            hidden_size = self.subject_hidden_dim,
            num_layers = self.subject_lstm_nlayers,
            batch_first = True,
            bidirectional = subject_lstm_bidirectional
        )

        # Speaker
        self.speaker_vocab_dim = speaker_vocab_dim
        self.speaker_embed_dim = speaker_embed_dim

        self.speaker_embedding = nn.Embedding(self.speaker_vocab_dim, self.speaker_embed_dim)

        # Speaker Position
        self.speaker_pos_vocab_dim = speaker_pos_vocab_dim
        self.speaker_pos_embed_dim = speaker_pos_embed_dim
        self.speaker_pos_lstm_nlayers = speaker_pos_lstm_nlayers
        self.speaker_pos_lstm_num_direction = 2 if speaker_pos_lstm_bidirectional else 1
        self.speaker_pos_hidden_dim = speaker_pos_hidden_dim

        self.speaker_pos_embedding = nn.Embedding(self.speaker_pos_vocab_dim, self.speaker_pos_embed_dim)
        self.speaker_pos_lstm = nn.LSTM(
            input_size = self.speaker_pos_embed_dim,
            hidden_size = self.speaker_pos_hidden_dim,
            num_layers = self.speaker_pos_lstm_nlayers,
            batch_first = True,
            bidirectional = speaker_pos_lstm_bidirectional
        )

        # State
        self.state_vocab_dim = state_vocab_dim
        self.state_embed_dim = state_embed_dim

        self.state_embedding = nn.Embedding(self.state_vocab_dim, self.state_embed_dim)

        # Party
        self.party_vocab_dim = party_vocab_dim
        self.party_embed_dim = party_embed_dim

        self.party_embedding = nn.Embedding(self.party_vocab_dim, self.party_embed_dim)

        # Context
        self.context_vocab_dim = context_vocab_dim
        self.context_embed_dim = context_embed_dim
        self.context_lstm_nlayers = context_lstm_nlayers
        self.context_lstm_num_direction = 2 if context_lstm_bidirectional else 1
        self.context_hidden_dim = context_hidden_dim

        self.context_embedding = nn.Embedding(self.context_vocab_dim, self.context_embed_dim)
        self.context_lstm = nn.LSTM(
            input_size = self.context_embed_dim,
            hidden_size = self.context_hidden_dim,
            num_layers = self.context_lstm_nlayers,
            batch_first = True,
            bidirectional = context_lstm_bidirectional
        )

        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(len(self.statement_kernel_size) * self.statement_kernel_num
                            + self.subject_lstm_nlayers * self.subject_lstm_num_direction
                            + self.speaker_embed_dim
                            + self.speaker_pos_lstm_nlayers * self.speaker_pos_lstm_num_direction
                            + self.state_embed_dim
                            + self.party_embed_dim
                            + self.context_lstm_nlayers * self.context_lstm_num_direction,
                            6)

    def forward(self,
                sample):
        statement = Variable(sample.statement).unsqueeze(0)
        subject = Variable(sample.subject).unsqueeze(0)
        speaker = Variable(sample.speaker).unsqueeze(0)
        speaker_pos = Variable(sample.speaker_pos).unsqueeze(0)
        state = Variable(sample.state).unsqueeze(0)
        party = Variable(sample.party).unsqueeze(0)
        context = Variable(sample.context).unsqueeze(0)

        batch = 1 # Current support one sample per time
                  # TODO: Increase batch number

        # Statement
        statement_ = self.statement_embedding(statement).unsqueeze(0) # 1*W*D -> 1*1*W*D
        statement_ = [F.relu(conv(statement_)).squeeze(3) for conv in self.statement_convs] # 1*1*W*1 -> 1*Co*W x [len(convs)]
        statement_ = [F.max_pool1d(i, i.size(2)).squeeze(2) for i in statement_] # 1*Co*1 -> 1*Co x len(convs)
        statement_ = torch.cat(statement_, 1)  # 1*len(convs)

        # Subject
        subject_ = self.subject_embedding(subject) # 1*W*D
        _, (subject_, _) = self.subject_lstm(subject_) # 1*(layer x dir)*hidden
        subject_ = F.max_pool1d(subject_, self.subject_hidden_dim).view(1, -1) # 1*(layer x dir)*1 -> 1*(layer x dir)

        # Speaker
        speaker_ = self.speaker_embedding(speaker).squeeze(0) # 1*1*D -> 1*D

        # Speaker Position
        speaker_pos_ = self.speaker_pos_embedding(speaker_pos)
        _, (speaker_pos_, _) = self.speaker_pos_lstm(speaker_pos_)
        speaker_pos_ = F.max_pool1d(speaker_pos_, self.speaker_pos_hidden_dim).view(1, -1)

        # State
        state_ = self.state_embedding(state).squeeze(0)

        # Party
        party_ = self.party_embedding(party).squeeze(0)

        # Context
        context_ = self.context_embedding(context)
        _, (context_, _) = self.context_lstm(context_)
        context_ = F.max_pool1d(context_, self.context_hidden_dim).view(1, -1)

        # Concatenate
        features = torch.cat((statement_, subject_, speaker_, speaker_pos_, state_, party_, context_), 1)
        features = self.dropout(features)
        features = self.fc(features)

        return features

In [ ]:

num_to_label = ['pants-fire',
                'false',
                'barely-true',
                'half-true',
                'mostly-true',
                'true']

label_to_number = {
	'pants-fire': 0,
	'false': 1,
	'barely-true': 2,
	'half-true': 3,
	'mostly-true': 4,
	'true': 5
}

def valid(valid_samples, word2num, model):
    acc = 0
    for sample in valid_samples:
        prediction = model(sample)
        prediction = int(np.argmax(prediction.data.numpy()))
        if prediction == sample.label:
            acc += 1
    acc /= len(valid_samples)
    print('  Validation Accuracy: '+str(acc))

def find_word(word2num, token):
    if token in word2num:
        return word2num[token]
    else:
        return word2num['<unk>']
    
def test_data_prepare(test_file, word2num, phase):
    test_input = open(test_file, 'rb')
    test_data = test_input.read().decode('utf-8')
    test_input.close()

    statement_word2num = word2num[0]
    subject_word2num = word2num[1]
    speaker_word2num = word2num[2]
    speaker_pos_word2num = word2num[3]
    state_word2num = word2num[4]
    party_word2num = word2num[5]
    context_word2num = word2num[6]

    test_samples = []

    for line in test_data.strip().split('\n'):
        tmp = line.strip().split('\t')
        while len(tmp) != 8:
            tmp.append('')
        if phase == 'test':
            p = DataSample('test', tmp[0], tmp[1], tmp[2], tmp[3], tmp[4], tmp[5], tmp[6])
        elif phase == 'valid':
            p = DataSample(tmp[0], tmp[1], tmp[2], tmp[3], tmp[4], tmp[5], tmp[6], tmp[7])

        for i in range(len(p.statement)):
            p.statement[i] = find_word(statement_word2num, p.statement[i])
        for i in range(len(p.subject)):
            p.subject[i] = find_word(subject_word2num, p.subject[i])
        p.speaker = find_word(speaker_word2num, p.speaker)
        for i in range(len(p.speaker_pos)):
            p.speaker_pos[i] = find_word(speaker_pos_word2num, p.speaker_pos[i])
        p.state = find_word(state_word2num, p.state)
        p.party = find_word(party_word2num, p.party)
        for i in range(len(p.context)):
            p.context[i] = find_word(context_word2num, p.context[i])

        test_samples.append(p)

    return test_samples

def test(test_file, test_output, word2num, model, use_cuda = False):
    test_samples = test_data_prepare(test_file, word2num, 'test')
    dataset_to_variable(test_samples, use_cuda)
    out = open(test_output, 'w')

    for sample in test_samples:
        prediction = model(sample)
        prediction = int(np.argmax(prediction.data.numpy()))
        out.write(num_to_label[prediction]+'\n')

    out.close()

In [ ]:
def dataset_to_variable(dataset, use_cuda):
	for i in range(len(dataset)):
		dataset[i].statement = torch.LongTensor(dataset[i].statement)
		dataset[i].subject = torch.LongTensor(dataset[i].subject)
		dataset[i].speaker = torch.LongTensor([dataset[i].speaker])
		dataset[i].speaker_pos = torch.LongTensor(dataset[i].speaker_pos)
		dataset[i].state = torch.LongTensor([dataset[i].state])
		dataset[i].party = torch.LongTensor([dataset[i].party])
		dataset[i].context = torch.LongTensor(dataset[i].context)
		if use_cuda:
			dataset[i].statement.cuda()
			dataset[i].subject.cuda()
			dataset[i].speaker.cuda()
			dataset[i].speaker_pos.cuda()
			dataset[i].state.cuda()
			dataset[i].party.cuda()
			dataset[i].context.cuda()

def train(train_samples,
          valid_samples,
          word2num,
          lr = 0.001,
          epoch = 5,
          use_cuda = False):

    print('Training...')

    # Prepare training data
    print('  Preparing training data...')
    statement_word2num = word2num[0]
    subject_word2num = word2num[1]
    speaker_word2num = word2num[2]
    speaker_pos_word2num = word2num[3]
    state_word2num = word2num[4]
    party_word2num = word2num[5]
    context_word2num = word2num[6]

    train_data = train_samples
    dataset_to_variable(train_data, use_cuda)
    valid_data = valid_samples
    dataset_to_variable(valid_data, use_cuda)

    # Construct model instance
    print('  Constructing network model...')
    model = Net(len(statement_word2num),
                len(subject_word2num),
                len(speaker_word2num),
                len(speaker_pos_word2num),
                len(state_word2num),
                len(party_word2num),
                len(context_word2num))
    if use_cuda: model.cuda()

    # Start training
    print('  Start training')

    optimizer = optim.Adam(model.parameters(), lr = lr)
    model.train()

    step = 0
    display_interval = 2000

    for epoch_ in range(epoch):
        print('  ==> Epoch '+str(epoch_)+' started.')
        random.shuffle(train_data)
        total_loss = 0
        for sample in train_data:

            optimizer.zero_grad()

            prediction = model(sample)
            label = Variable(torch.LongTensor([sample.label]))
            loss = F.cross_entropy(prediction, label)
            loss.backward()
            optimizer.step()

            step += 1
            if step % display_interval == 0:
                print('    ==> Iter: '+str(step)+' Loss: '+str(loss))

            total_loss += loss.data.numpy()

        print('  ==> Epoch '+str(epoch_)+' finished. Avg Loss: '+str(total_loss/len(train_data)))

        valid(valid_data, word2num, model)

    return model

In [ ]:
# using Bert


# set random seeds to ensure reproducibility
SEED = 1234
torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

# set batch size
BATCH_SIZE = 64

# set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# define fields
TEXT = data.Field(tokenize='spacy', include_lengths=True)
LABEL = data.LabelField(dtype=torch.float)

# make splits for data liar dataset


In [9]:
# using deep learning torch

class NewsDataset(Dataset):
    def __init__(self, csv_file, transform=None):
        self.data = pd.read_csv(csv_file, on_bad_lines='skip', sep='\t', header=None)
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        if torch.is_tensor(idx):
            idx = idx.tolist()

        text = self.data.iloc[idx, 2]
        label = self.data.iloc[idx, 1]
        #print(text, label)

        if self.transform:
            text = self.transform(text)

        return text, label

class ToTensor(object):
    """Convert ndarrays in sample to Tensors."""

    def __call__(self, text):
        return torch.from_numpy(text).float()

class TextCNN(nn.Module):
    def __init__(self, vocab_size, embedding_dim, n_filters, filter_sizes, output_dim, dropout):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embedding_dim)

        self.convs = nn.ModuleList([
                                    nn.Conv2d(in_channels = 1,
                                              out_channels = n_filters,
                                              kernel_size = (fs, embedding_dim))
                                    for fs in filter_sizes
                                    ])

        self.fc = nn.Linear(len(filter_sizes) * n_filters, output_dim)

        self.dropout = nn.Dropout(dropout)

    def forward(self, text):

        #text = [sent len, batch size]

        #text = text.permute(1, 0)

        #text = [batch size, sent len]

        #transform text to tensor

        text = text.unsqueeze(1)

        embedded = self.embedding(text)

        #embedded = [batch size, sent len, emb dim]

        embedded = embedded.unsqueeze(1)

        #embedded = [batch size, 1, sent len, emb dim]

        conved = [F.relu(conv(embedded)).squeeze(3) for conv in self.convs]

        #conv_n = [batch size, n_filters, sent len - filter_sizes[n] + 1]

        pooled = [F.max_pool1d(conv, conv.shape[2]).squeeze(2) for conv in conved]

        #pooled_n = [batch size, n_filters]

        cat = self.dropout(torch.cat(pooled, dim = 1))

        #cat = [batch size, n_filters * len(filter_sizes)]

        return self.fc(cat)

def binary_accuracy(preds, y):
    """
    Returns accuracy per batch, i.e. if you get 8/10 right, this returns 0.8, NOT 8
    """

    #round predictions to the closest integer
    rounded_preds = torch.round(torch.sigmoid(preds))
    correct = (rounded_preds == y).float()
    acc = correct.sum()/len(correct)
    return acc

def train(model, iterator, optimizer, criterion):

        epoch_loss = 0
        epoch_acc = 0

        model.train()

        for text, labels in iterator:

            optimizer.zero_grad()

            predictions = model(text).squeeze(1)

            loss = criterion(predictions, labels)

            acc = binary_accuracy(predictions, labels)

            loss.backward()

            optimizer.step()

            epoch_loss += loss.item()
            epoch_acc += acc.item()

        return epoch_loss / len(iterator), epoch_acc / len(iterator)


def evaluate(model, iterator, criterion):
    epoch_loss = 0
    epoch_acc = 0

    model.eval()

    with torch.no_grad():

        for text, labels in iterator:

            predictions = model(text).squeeze(1)

            loss = criterion(predictions, labels)

            acc = binary_accuracy(predictions, labels)

            epoch_loss += loss.item()
            epoch_acc += acc.item()

    return epoch_loss / len(iterator), epoch_acc / len(iterator)


In [10]:
# Set hyperparameters
size_of_vocab = len(tfidf_vectorizer.vocabulary_) + 1  # Add 1 for the <UNK> token
embedding_dim = 100
n_filters = 100
filter_sizes = [3, 4, 5]
output_dim = 1
dropout = 0.5
batch_size = 64

# Instantiate the model
model = TextCNN(size_of_vocab, embedding_dim, n_filters, filter_sizes, output_dim, dropout)
model = model.to(device)

glove = vocab.GloVe(name='6B', dim=100)

# Get the embeddings corresponding to words in both GloVe and your TF-IDF vocabulary
embedding_vectors = []
words_not_found = 0
unk_token = torch.zeros(embedding_dim)  # Placeholder for <UNK> token

for word in tfidf_vectorizer.vocabulary_:
    if word in glove.stoi:
        embedding_vectors.append(glove[word])
    else:
        # For words not found in GloVe, use the <UNK> token
        embedding_vectors.append(unk_token)
        words_not_found += 1

pretrained_embeddings = torch.stack(embedding_vectors)

# Add the <UNK> token vector to pretrained_embeddings
pretrained_embeddings = torch.cat((pretrained_embeddings, unk_token.unsqueeze(0)), dim=0)

# Print the count of words not found in GloVe
print(f"Words not found in GloVe embeddings: {words_not_found}")

# Ensure the dimensions match
if pretrained_embeddings.size(0) != size_of_vocab:
    raise ValueError("Dimensions of embeddings do not match the size of vocabulary.")

model.embedding.weight.data.copy_(pretrained_embeddings)

# Initialize the optimizer and criterion
optimizer = optim.Adam(model.parameters(), lr=0.0001)
criterion = nn.BCEWithLogitsLoss()

#split the data into train and test sets using test.tsv and train.tsv
train_dataset = NewsDataset(csv_file='/content/liar_dataset/train.tsv')
test_dataset = NewsDataset(csv_file='/content/liar_dataset/test.tsv')

#split the train dataset into train and validation sets
train_data, valid_data = torch.utils.data.random_split(train_dataset, [len(train_dataset)-1000, 1000])

#initialize the dataloaders
train_iterator = DataLoader(train_data, shuffle=True, batch_size=batch_size)
valid_iterator = DataLoader(valid_data, batch_size=batch_size)
test_iterator = DataLoader(test_dataset, batch_size=batch_size)


Words not found in GloVe embeddings: 581


In [11]:
#train the model
N_EPOCHS = 5

best_valid_loss = float('inf')

for epoch in range(N_EPOCHS):

        train_loss, train_acc = train(model, train_iterator, optimizer, criterion)
        valid_loss, valid_acc = evaluate(model, valid_iterator, criterion)

        if valid_loss < best_valid_loss:
            best_valid_loss = valid_loss
            torch.save(model.state_dict(), 'tut5-model.pt')

        print(f'Epoch: {epoch+1:02}')
        print(f'\tTrain Loss: {train_loss:.3f} | Train Acc: {train_acc*100:.2f}%')
        print(f'\t Val. Loss: {valid_loss:.3f} |  Val. Acc: {valid_acc*100:.2f}%')

#test the model
model.load_state_dict(torch.load('tut5-model.pt'))

test_loss, test_acc = evaluate(model, test_iterator, criterion)

print(f'Test Loss: {test_loss:.3f} | Test Acc: {test_acc*100:.2f}%')

TypeError: ignored